# Week 6 Exercise: Fine-Tuning an Open-Source Model with QLoRA

## "The Price Is Right" — Fine-Tuning Approach

Instead of using OpenAI's fine-tuning API (Day 5), we'll fine-tune an open-source model
(**Llama-3.2-1B**) using **QLoRA** (Quantized Low-Rank Adaptation) to predict product prices.

This notebook is designed to run on **Google Colab with a free T4 GPU**.

### Learning Objectives
- Load and prepare the curated pricing dataset from HuggingFace
- Apply 4-bit quantization with BitsAndBytes for memory-efficient training
- Fine-tune with LoRA adapters using SFTTrainer from the TRL library
- Evaluate the fine-tuned model against a baseline

In [ ]:
# Step 1: Install Dependencies
%pip install -q datasets peft bitsandbytes transformers trl accelerate torch

In [ ]:
# Step 2: Imports & HuggingFace Login

import os
import re
import torch
import numpy as np
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM
from google.colab import userdata
from huggingface_hub import login

# Login to HuggingFace — add your HF_TOKEN in Colab Secrets (key icon in left sidebar)
hf_token = userdata.get("HF_TOKEN")
login(hf_token)

In [ ]:
# Step 3: Load Dataset & Prepare Prompts

BASE_MODEL = "meta-llama/Llama-3.2-1B"
QUESTION = "What does this cost to the nearest dollar?"
PREFIX = "Price is $"

# Load the pre-processed lite dataset from HuggingFace
ds = load_dataset("ed-donner/items_lite")

print(f"Train: {len(ds['train']):,}, Validation: {len(ds['validation']):,}, Test: {len(ds['test']):,}")

# Format each item into a prompt string for fine-tuning
def make_prompt(row):
    text = row["summary"] or row["title"]
    price = round(float(row["price"]))
    row["text"] = f"{QUESTION}\n\n{text}\n\n{PREFIX}{price}.00"
    return row

# Use a subset to keep training fast on Colab free tier
train_ds = ds["train"].select(range(500)).map(make_prompt)
val_ds = ds["validation"].select(range(50)).map(make_prompt)
test_ds = ds["test"].select(range(100)).map(make_prompt)

print(f"\nFine-tuning with {len(train_ds)} train, {len(val_ds)} val, {len(test_ds)} test examples")
print(f"\nSample prompt:\n{train_ds[0]['text']}")

In [ ]:
# Step 4: Load Base Model with 4-bit Quantization + LoRA Config

# 4-bit quantization config for QLoRA
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

# Load the base model quantized
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto"
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

# LoRA adapter config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Base model loaded: {total_params:,} total params")
print(f"Device: {next(model.parameters()).device}")

In [ ]:
# Step 4b: Test Base Model BEFORE Fine-Tuning (Zero-Shot Baseline)
# This lets us measure how much fine-tuning actually improves the model

def extract_price(text):
    """Extract the first number after 'Price is $' from model output."""
    match = re.search(r"Price is \$(\d+(?:\.\d+)?)", text)
    if match:
        return float(match.group(1))
    match = re.search(r"\d+(?:\.\d+)?", text)
    return float(match.group()) if match else 0.0

def predict_price(item_text, mdl, tok):
    """Generate a price prediction for a single item."""
    prompt = f"{QUESTION}\n\n{item_text}\n\n{PREFIX}"
    inputs = tok(prompt, return_tensors="pt").to(mdl.device)
    with torch.no_grad():
        output = mdl.generate(**inputs, max_new_tokens=5, do_sample=False)
    decoded = tok.decode(output[0], skip_special_tokens=True)
    return extract_price(decoded)

print("Testing base model (before fine-tuning) on 20 test items...\n")
base_errors = []
for i in range(20):
    item = test_ds[i]
    text = item["summary"] or item["title"]
    actual = float(item["price"])
    predicted = predict_price(text, model, tokenizer)
    error = abs(predicted - actual)
    base_errors.append(error)
    print(f"  Actual: ${actual:>7.2f}  |  Base predicted: ${predicted:>7.2f}  |  Error: ${error:.2f}")

base_mae = np.mean(base_errors)
print(f"\nBase model MAE (before fine-tuning): ${base_mae:,.2f}")

In [ ]:
# Step 5: Fine-Tune with SFTTrainer

# Data collator — only trains on the response portion (after "Price is $")
response_template = PREFIX
collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)

# Training configuration
training_args = SFTConfig(
    output_dir="pricer-qlora",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    optimizer="paged_adamw_32bit",
    max_seq_length=256,
    dataset_text_field="text",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=25,
    report_to="none",
    seed=42,
)

# Create trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=lora_config,
    args=training_args,
    data_collator=collator,
)

# Train!
trainer.train()

In [ ]:
# Step 5b: Save the Fine-Tuned Adapter Weights

# Save locally
LOCAL_PATH = "pricer-qlora-adapter"
trainer.model.save_pretrained(LOCAL_PATH)
tokenizer.save_pretrained(LOCAL_PATH)
print(f"Adapter saved locally to '{LOCAL_PATH}/'")

# Optionally push to HuggingFace Hub — uncomment and set your username
# HF_USERNAME = "your-username"
# REPO_NAME = f"{HF_USERNAME}/pricer-qlora-llama-3.2-1b"
# trainer.model.push_to_hub(REPO_NAME)
# tokenizer.push_to_hub(REPO_NAME)
# print(f"Pushed to HuggingFace Hub: {REPO_NAME}")

In [ ]:
# Step 6: Evaluate the Fine-Tuned Model

GREEN = "\033[92m"
YELLOW = "\033[93m"
RED = "\033[91m"
RESET = "\033[0m"

errors = []
actuals = []
predictions = []

for i in range(len(test_ds)):
    item = test_ds[i]
    text = item["summary"] or item["title"]
    actual = float(item["price"])
    predicted = predict_price(text, model, tokenizer)
    error = abs(predicted - actual)
    errors.append(error)
    actuals.append(actual)
    predictions.append(predicted)

    # Color-coded output like the course evaluator
    if error < 40 or (actual > 0 and error / actual < 0.2):
        color = GREEN
    elif error < 80 or (actual > 0 and error / actual < 0.4):
        color = YELLOW
    else:
        color = RED
    print(f"{color}${error:.0f} ", end="")

# Baseline: always predict the average training price
train_prices = [float(row["price"]) for row in train_ds]
avg_price = sum(train_prices) / len(train_prices)
baseline_errors = [abs(avg_price - float(row["price"])) for row in test_ds]

print(f"\n\n{RESET}{'='*60}")
print(f"RESULTS on {len(test_ds)} test items:")
print(f"{'='*60}")
print(f"Fine-tuned model MAE:    ${np.mean(errors):,.2f}")
print(f"Base model MAE (pre-FT): ${base_mae:,.2f}")
print(f"Baseline (avg price) MAE: ${np.mean(baseline_errors):,.2f}")
print(f"\nFine-tuned model median error: ${np.median(errors):,.2f}")
print(f"Best predictions:  {sum(1 for e in errors if e < 20)} items within $20")
print(f"Good predictions:  {sum(1 for e in errors if e < 40)} items within $40")

In [ ]:
# Step 7: Visualization — Predicted vs Actual Prices

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# --- Scatter Plot: Predicted vs Actual ---
ax1 = axes[0]
colors = []
for pred, act in zip(predictions, actuals):
    err = abs(pred - act)
    if err < 40 or (act > 0 and err / act < 0.2):
        colors.append("green")
    elif err < 80 or (act > 0 and err / act < 0.4):
        colors.append("orange")
    else:
        colors.append("red")

ax1.scatter(actuals, predictions, c=colors, s=30, alpha=0.7)
max_val = max(max(actuals), max(predictions))
ax1.plot([0, max_val], [0, max_val], "b--", linewidth=1.5, label="Perfect prediction (y=x)")
ax1.set_xlabel("Actual Price ($)", fontsize=12)
ax1.set_ylabel("Predicted Price ($)", fontsize=12)
ax1.set_title(f"Fine-Tuned Model: Predicted vs Actual\nMAE = ${np.mean(errors):,.2f}", fontsize=13)
ax1.legend()
ax1.set_xlim(0, max_val * 1.05)
ax1.set_ylim(0, max_val * 1.05)

# --- Bar Chart: Model Comparison ---
ax2 = axes[1]
model_names = ["Baseline\n(Avg Price)", "Base Model\n(Zero-Shot)", "Fine-Tuned\n(QLoRA)"]
mae_values = [np.mean(baseline_errors), base_mae, np.mean(errors)]
bar_colors = ["gray", "orange", "green"]

bars = ax2.bar(model_names, mae_values, color=bar_colors, width=0.5)
for bar, val in zip(bars, mae_values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f"${val:,.2f}", ha="center", va="bottom", fontsize=12, fontweight="bold")

ax2.set_ylabel("Mean Absolute Error ($)", fontsize=12)
ax2.set_title("Model Comparison — Lower is Better", fontsize=13)
ax2.set_ylim(0, max(mae_values) * 1.25)

plt.tight_layout()
plt.show()

# --- Error Distribution Histogram ---
fig2, ax3 = plt.subplots(figsize=(12, 5))
ax3.hist(errors, bins=30, color="steelblue", edgecolor="white", alpha=0.8)
ax3.axvline(np.mean(errors), color="red", linestyle="--", linewidth=2, label=f"Mean Error = ${np.mean(errors):,.2f}")
ax3.axvline(np.median(errors), color="orange", linestyle="--", linewidth=2, label=f"Median Error = ${np.median(errors):,.2f}")
ax3.set_xlabel("Absolute Error ($)", fontsize=12)
ax3.set_ylabel("Count", fontsize=12)
ax3.set_title("Error Distribution of Fine-Tuned Model", fontsize=13)
ax3.legend(fontsize=11)
plt.tight_layout()
plt.show()